plot A function

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

DEFAULT_FILE_PATH = Path(
    "/Users/sam/Documents/Oxford/Physics/sloppiness/"
    "circadian/mut_project_updates/figures/RNA/L_30/"
    "plot_a/plot_a_files/Plot_A_RNA_L30_Billion_Samples.csv"
)

def plot_A(ax, file_path: Path | str | None = None):
    """
    RNA L=30 — Plot A
    Fixed bin width = 2.5 with adaptive log inset floor.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis to draw the plot on.
    file_path : Path | str | None
        Path to CSV. If None, uses DEFAULT_FILE_PATH.
    Returns
    -------
    ax : matplotlib.axes.Axes
        The axis with the plot.
    """

    # -----------------------
    # Helpers: load dataset
    # -----------------------
    def load_rna(fp):
        fp = Path(fp)
        print(f"\nLoading from: {fp}")

        df = pd.read_csv(fp, sep=r"\s+|\t|,", engine="python")
        df.columns = df.columns.str.strip()

        # keep only Rand and 0muts
        df = df[df["Status"].isin(["Rand", "0muts"])]

        df_rand = df[df["Status"] == "Rand"].copy()
        df_p = df[df["Status"] == "0muts"].copy()

        # compute relative frequencies
        df_rand["RelFreq"] = df_rand["Freq"] / df_rand["Freq"].sum()
        df_p["RelFreq"] = df_p["Freq"] / df_p["Freq"].sum()
        return df_rand, df_p
    

    # -----------------------
    # Load data & shared range
    # -----------------------
    if file_path is None:
        file_path = DEFAULT_FILE_PATH

    df_rand, df_p = load_rna(file_path)

    # shared min / max over Complexity
    all_vals = np.concatenate([df_rand["Complexity"].values, df_p["Complexity"].values])
    min_val = all_vals.min()
    max_val = all_vals.max()

    # ===================================
    # Fixed bin width
    # ===================================
    bin_width = 2.5
    bins = np.arange(min_val, max_val + bin_width, bin_width)
    n_bins = len(bins) - 1

    print(f"\n[RNA30 Plot A] Bin width = {bin_width}")
    print(f"[RNA30 Plot A] Number of bins = {n_bins}")

    # Histogram (PMF via weights)
    rand_counts, bin_edges = np.histogram(
        df_rand["Complexity"],
        bins=bins,
        weights=df_rand["RelFreq"]
    )

    p_counts, _ = np.histogram(
        df_p["Complexity"],
        bins=bins,
        weights=df_p["RelFreq"]
    )

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    # ===================================
    # Print distribution statistics
    # ===================================
    g_dist_complexity_mean = (
        (df_rand["Complexity"] * df_rand["Freq"]).sum()
        / df_rand["Freq"].sum()
    )

    p_dist_complexity_mean = (
        (df_p["Complexity"] * df_p["Freq"]).sum()
        / df_p["Freq"].sum()
    )
    print(f"G-dist mean complexity: {g_dist_complexity_mean:.6f}")
    print(f"P-dist mean complexity: {p_dist_complexity_mean:.6f}")
    #print(f"Number of samples in G-dist: {len(df_rand)}")
    #print(f"Number of samples in P-dist: {len(df_p)}")


    # ===================================
    # Print distribution statistics
    # ===================================

    # ---- G distribution ----
    g_weights = df_rand["Freq"].values
    g_values = df_rand["Complexity"].values

    g_mean = (g_values * g_weights).sum() / g_weights.sum()
    g_var = (g_weights * (g_values - g_mean)**2).sum() / g_weights.sum()
    g_std = np.sqrt(g_var)

    # ---- P distribution ----
    p_weights = df_p["Freq"].values
    p_values = df_p["Complexity"].values

    p_mean = (p_values * p_weights).sum() / p_weights.sum()
    p_var = (p_weights * (p_values - p_mean)**2).sum() / p_weights.sum()
    p_std = np.sqrt(p_var)

    print(f"G-dist mean complexity: {g_mean:.6f}")
    print(f"G-dist std complexity : {g_std:.6f}")
    print()
    print(f"P-dist mean complexity: {p_mean:.6f}")
    print(f"P-dist std complexity : {p_std:.6f}")

    # ===================================
    # Adaptive epsilon floor
    # ===================================
    all_nonzero = np.concatenate([
        rand_counts[rand_counts > 0],
        p_counts[p_counts > 0]
    ])

    if all_nonzero.size > 0:
        eps = np.min(all_nonzero) * 0.1
    else:
        # fallback if both arrays are zero (avoid zero eps)
        eps = 1e-12

    print(f"[RNA30 Plot A] Epsilon floor value = {eps:.3e}")

    rand_safe = np.clip(rand_counts, eps, None)
    p_safe = np.clip(p_counts, eps, None)

    # ===================================
    # Main Plot
    # ===================================
    ax.plot(bin_centers,
            rand_counts,
            marker='o',
            color="orange",
            label="G-dist")

    ax.fill_between(bin_centers,
                    rand_counts,
                    0,
                    color="orange",
                    alpha=0.18)

    ax.plot(bin_centers,
            p_counts,
            marker='o',
            color="blue",
            label="P-dist")

    ax.fill_between(bin_centers,
                    p_counts,
                    0,
                    color="blue",
                    alpha=0.15)

    ax.set_xlabel("Complexity (Lempel-Ziv)")
    ax.set_ylabel("Relative Frequency")

    ax.legend(frameon=False,
              edgecolor="black",
              loc="upper left")

    ax.tick_params(labelsize=12)
    ax.set_axisbelow(True)

    # ===================================
    # Inset
    # ===================================
    ax_inset = inset_axes(ax, width="22%", height="22%", loc="upper right")

    ax_inset.plot(bin_centers,
                  rand_safe,
                  color="orange")

    ax_inset.plot(bin_centers,
                  p_safe,
                  color="blue")

    ax_inset.set_yscale("log")
    ax_inset.set_xlabel("Complexity", fontsize=9)
    ax_inset.set_ylabel("Log Frequency", fontsize=9)

    ax_inset.tick_params(labelsize=8)
    #ax_inset.grid(which="both",
    #              axis="y",
    #              alpha=0.3,
    #              linestyle="--")

    return ax

In [ ]:
# Weighted mean complexity
mean_complexity_rand = (
    (df_rand["Complexity"] * df_rand["Freq"]).sum()
    / df_rand["Freq"].sum()
)

mean_complexity_0muts = (
    (df_0muts["Complexity"] * df_0muts["Freq"]).sum()
    / df_0muts["Freq"].sum()
)

print(f"Weighted mean complexity (Rand): {mean_complexity_rand:.6f}")
print(f"Weighted mean complexity (0muts): {mean_complexity_0muts:.6f}")

plot B function

In [ ]:
def plot_B_RNA30(ax):
    """
    RNA L=30 — Plot B
    Entropy and global scaled complexity vs sample size.
    """

    import pandas as pd
    from pathlib import Path

    p30 = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/RNA/L_30/"
        "plot_b/plot_b_files/RNA_L30_global_local_sampling.csv"
    )

    df30 = pd.read_csv(p30)

    rename_map = {
        "Samples": "sample_size",
        "Entropy": "entropy",
        "GlobalScaled": "scaled_global"
    }

    df30 = df30.rename(columns=rename_map)

    df30["sample_size"] = pd.to_numeric(df30["sample_size"], errors="coerce")
    df30["entropy"] = pd.to_numeric(df30["entropy"], errors="coerce")
    df30["scaled_global"] = pd.to_numeric(df30["scaled_global"], errors="coerce")

    df30.sort_values("sample_size", inplace=True)

    N = df30["sample_size"].values
    S = df30["entropy"].values
    Kg = df30["scaled_global"].values

    ax.semilogx(N, S, "o-",
                label=r"$S$", color="green")

    ax.semilogx(N, Kg, "o-",
                label=r"$\langle \tilde{K}_{g} \rangle$", color="orange")

    ax.set_xlabel("Number of sampled genotypes (N, log scale)")
    ax.set_ylabel(r'Entropy (S) and scaled $\langle \tilde{K}_{g} \rangle$')
    ax.set_xscale("log")
    ax.minorticks_off()
    ax.set_xlim(5, 2_000_000_000)

    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    ax.legend(frameon=False)

plot C Function

In [ ]:
def plot_C(ax):
    """
    RNA L=30 — Plot C
    Mean complexity vs Shannon entropy (linear regression).
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path

    data_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/RNA/"
        "plot_c/plot_c_files/Plot_C_data_RNA_N_Million_Samples.xlsx"
    )

    df = pd.read_excel(data_path)

    df["L"] = pd.to_numeric(df["L"], errors="coerce")
    df["S"] = pd.to_numeric(df["S"], errors="coerce")
    df["Kg"] = pd.to_numeric(df["Kg"], errors="coerce")

    df = df.dropna()

    entropy_vals = df["S"].values
    kg_vals = df["Kg"].values
    labels = df["L"].astype(int).astype(str).values

    # Linear regression
    m, b = np.polyfit(entropy_vals, kg_vals, 1)
    fit_line = m * entropy_vals + b
    sign = "-" if b < 0 else "+"
    eq_label = rf"$\langle \tilde{{K}}_{{g}} \rangle = {m:.2f}\,S {sign} {abs(b):.2f}$"

    ax.scatter(entropy_vals, kg_vals, color='darkorange')
    ax.plot(entropy_vals, fit_line, color='orange', label=eq_label)

    for xi, yi, lbl in zip(entropy_vals, kg_vals, labels):
        ax.text(xi, yi, lbl, ha='center', va='bottom', fontsize=11)

    ax.set_xlabel("Shannon Entropy (S)")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")
    ax.margins(x=0.05, y=0.05)

    ax.legend(fontsize=12,frameon=False)

plot D Function

In [ ]:
def plot_D(ax):
    """
    RNA L=30 — Plot D
    Mean G1..G5 vs mutation number.
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path

    csv_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
        "mut_project_updates/figures/RNA/L_30/plot_d/plot_d_files/"
        "RNA_L30_k_scaled_1_for_plot_D.csv"
    )

    marker = "o"
    marker_size = 6

    # --------------------------------
    # Load CSV
    # --------------------------------
    df = pd.read_csv(csv_path)
    if df.empty:
        raise RuntimeError(f"No data found in {csv_path}")

    first_col = df.columns[0]
    first_col_lower = first_col.lower()

    if any(k in first_col_lower for k in ("mut", "number", "n", "k")):
        x_vals = df.iloc[:, 0].to_numpy()
        data_df = df.iloc[:, 1:].copy()
    else:
        x_vals = np.arange(len(df))
        data_df = df.copy()

    # Find MeanG* columns
    meang_cols = [
        c for c in data_df.columns
        if str(c).lower().startswith("meang")
    ]

    if not meang_cols:
        meang_cols = [
            c for c in data_df.columns
            if any(s in str(c).lower() for s in ("g1","g2","g3","g4","g5"))
        ]

    if not meang_cols:
        raise RuntimeError("No MeanG* columns found in CSV.")

    # Prepare arrays
    series_vals = []
    for c in meang_cols:
        vals = pd.to_numeric(data_df[c], errors="coerce").to_numpy()
        series_vals.append(vals)

    series_vals = np.array(series_vals)

    # x positions
    x_positions = np.arange(len(x_vals))

    if np.issubdtype(x_vals.dtype, np.number):
        x_tick_labels = [
            str(int(x)) if float(x).is_integer() else f"{x:.3g}"
            for x in x_vals
        ]
    else:
        x_tick_labels = [str(x) for x in x_vals]

    # --------------------------------
    # Plot
    # --------------------------------
    palette = [
    "#E8B5D6",  # very light pink (G1)
    "#CC79A7",  # soft rose (G2)
    "#A63478",  # medium raspberry (G3)
    "#7A1F5C",  # dark wine (G4)
    "#4A0D3A",  # very dark plum (G5)
]
    
    for i, vals in enumerate(series_vals):
        ax.plot(x_positions,
                vals,
                marker=marker,
                label=f"G{i+1}",
                color=palette[i])

    ax.set_xlabel("Number of mutations")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")

    # show every other tick
    ax.set_xticks(x_positions[::2])
    ax.set_xticklabels(x_tick_labels[::2])

    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    # Reverse legend (G5 → G1)
    handles, legend_labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], legend_labels[::-1],
              loc="upper right",fontsize=10,frameon=False)

    # 0.5 tick spacing margin
    if len(x_positions) > 1:
        spacing = x_positions[1] - x_positions[0]
    else:
        spacing = 1.0

    ax.set_xlim(x_positions[0] - 0.5 * spacing,
                x_positions[-1] + 0.5 * spacing)

create master figure

In [ ]:
import matplotlib.pyplot as plt

# ===================================
# Create master 2x2 figure
# ===================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Flatten for easier iteration
axes = axes.flatten()

# Call your plotting functions
plot_A(axes[0])
plot_B_RNA30(axes[1])
plot_C(axes[2])
plot_D(axes[3])

# ===================================
# Panel labels underneath
# ===================================
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for ax, label in zip(axes, panel_labels):
    ax.text(
        0.5,               # centered horizontally
        -0.14,             # below axis
        label,
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=11
    )

# Improve spacing
plt.tight_layout()
plt.subplots_adjust(hspace=0.28, wspace=0.25)

plt.show()